# Toolshed: Scale Tool-Equipped Agents with Advanced RAG-Tool Fusion - Hands-on Build

**Date:** 2026-06-09  
**Explainer:** [../toolshed.md](../toolshed.md)  
**Paper:** [arXiv:2410.14594](https://arxiv.org/abs/2410.14594)

---

## Problem

A wealth-management research copilot has a catalog of **60+ analytics tools** (rebalancing,
tax-loss harvesting, risk attribution, NPV, factor exposure, options Greeks, FX conversion, ...).
Stuffing all 60 into every prompt blows past the model's reliable tool limit and causes
mis-selection. Build a **Toolshed retrieval layer**: enrich and index the catalog, decompose
and expand each advisor query, retrieve + rerank down to a small `top-k`, and hand only those
tools to the agent - measuring **Recall@k** as you vary catalog size (`tool-M`) and threshold (`top-k`).

> WARNING: **Synthetic / public data only.** Decision-support, not advice. No real client data or MNPI.

In [ ]:
import os
import time
import json
import operator
from typing import TypedDict, Annotated, List, Dict, Any, Optional, Literal

import numpy as np
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field

# LangChain messages / prompts / tools
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel

# LangChain model providers + embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_mistralai import ChatMistralAI

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Send

# Verify required API keys
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set'
assert os.getenv('MISTRAL_API_KEY'), 'MISTRAL_API_KEY not set'
assert os.getenv('LANGSMITH_API_KEY'), 'LANGSMITH_API_KEY not set'

In [ ]:
# LLM instances - one per provider
llmOpenAI = ChatOpenAI(model='gpt-5.4-mini', temperature=0.0)
llmAnthropic = ChatAnthropic(model='claude-haiku-4-5', temperature=0.0)
llmMistral = ChatMistralAI(model='mistral-medium-3-5', temperature=0.0)

# Embedder for the Toolshed Knowledge Base (out-of-the-box, no fine-tuning - per the paper)
embedder = OpenAIEmbeddings(model='text-embedding-3-large')

## Synthetic data - a toy wealth-management tool catalog

A tiny stand-in for the 60+ tool catalog. Each tool is a `{name, description, args}` dict.
We also define a few **golden** query->tool pairings to score Recall@k later.

In [ ]:
# A small, deliberately overlapping-ish WM tool catalog (extend toward 60+ to stress the system).
TOOL_CATALOG: List[Dict[str, Any]] = [
    {'name': 'rebalance_portfolio', 'description': 'Rebalance a portfolio back to its target asset allocation weights.', 'args': {'account_id': 'str', 'target_weights': 'dict[str,float]'}},
    {'name': 'tax_loss_harvest', 'description': 'Identify lots with unrealized losses to harvest for tax purposes.', 'args': {'account_id': 'str', 'min_loss_usd': 'float'}},
    {'name': 'risk_attribution', 'description': 'Decompose portfolio risk by asset class and factor.', 'args': {'account_id': 'str'}},
    {'name': 'compute_npv', 'description': 'Compute net present value of a cash-flow stream given a discount rate.', 'args': {'cashflows': 'list[float]', 'rate': 'float'}},
    {'name': 'factor_exposure', 'description': 'Report exposure of holdings to value, momentum, size, quality factors.', 'args': {'account_id': 'str'}},
    {'name': 'options_greeks', 'description': 'Compute delta, gamma, theta, vega for an options position.', 'args': {'ticker': 'str', 'strike': 'float', 'expiry': 'str'}},
    {'name': 'fx_convert', 'description': 'Convert an amount between two currencies at the latest rate.', 'args': {'amount': 'float', 'from_ccy': 'str', 'to_ccy': 'str'}},
    {'name': 'get_price_history', 'description': 'Fetch historical daily close prices for a ticker.', 'args': {'ticker': 'str', 'lookback_days': 'int'}},
    {'name': 'sharpe_ratio', 'description': 'Compute the annualized Sharpe ratio of a return series.', 'args': {'returns': 'list[float]', 'rf_rate': 'float'}},
    {'name': 'sector_allocation', 'description': 'Break down portfolio market value by GICS sector.', 'args': {'account_id': 'str'}},
    {'name': 'dividend_forecast', 'description': 'Project expected dividend income over the next 12 months.', 'args': {'account_id': 'str'}},
    {'name': 'var_estimate', 'description': 'Estimate 1-day 95% Value-at-Risk for the portfolio.', 'args': {'account_id': 'str'}},
]

# Golden pairings: query -> set of correct tool names (for Recall@k).
GOLDEN: List[Dict[str, Any]] = [
    {'query': 'Rebalance the Smith account and harvest any tax losses over $1k.', 'tools': {'rebalance_portfolio', 'tax_loss_harvest'}},
    {'query': 'What is the NPV of these cash flows at 6%, and convert the result to EUR?', 'tools': {'compute_npv', 'fx_convert'}},
    {'query': 'Show me the risk breakdown and 1-day VaR for the Lee portfolio.', 'tools': {'risk_attribution', 'var_estimate'}},
]

print(f'tool-M (catalog size) = {len(TOOL_CATALOG)}')

## Coverage Map - complete every section below to cover the whole paper

| # | Paper aspect | Role in this build | Section |
|---|---|---|---|
| 1 | **Toolshed Knowledge Base** (vector store + metadata dict) | Store enriched tool docs; map display name -> real callable | S1 |
| 2 | **Pre-retrieval enrichment** (name, description, arg schema, synthetic Qs, key topics) | Turn each catalog tool into a rich, embeddable document | S2 |
| 3 | **Indexing / embedding** | Embed enriched docs into the KB | S2 |
| 4 | **Intra-retrieval: query rewrite** | Fix pronouns/typos, use chat history | S3 |
| 5 | **Intra-retrieval: query decomposition / planning** | Split a multi-part advisor query into sub-steps | S4 |
| 6 | **Intra-retrieval: multi-query expansion** | Generate diverse phrasings per sub-step | S5 |
| 7 | **Retrieval** | Query the KB per variant -> top-k candidates each | S6 |
| 8 | **Post-retrieval: rerank + dedupe** | Condense the candidate pile to a final top-k | S7 |
| 9 | **Post-retrieval: self-reflection (Self-RAG)** | Agent judges sufficiency; re-query if a tool is missing | S8 |
| 10 | **tool-M x top-k trade-off + Recall@k** | Measure recall vs catalog size and threshold | S9 |
| 11 | **Final agent tool-calling** | Equip only the top-k tools; let the agent answer | S10 |
| 12 | **Wire-up into one graph** | Assemble all phases into a runnable LangGraph | S11 |

### S1 . Toolshed Knowledge Base

**What:** a vector database of tools plus a metadata dictionary that maps each tool's embedded
display name back to its real callable. **Maps to:** the store that holds our 12 WM tools and
resolves a retrieved doc to the actual Python function the agent will call.

In [ ]:
class ToolshedKB:
    """Vector store of enriched tool documents + name->callable metadata dict."""

    def __init__(self, embedder):
        self.embedder = embedder
        self.docs: List[str] = []            # enriched text per tool
        self.vectors: List[np.ndarray] = []  # embeddings, aligned with self.docs
        self.names: List[str] = []           # true tool name per doc
        self.metadata: Dict[str, Dict] = {}  # true_name -> tool spec / callable

    def add(self, true_name: str, enriched_doc: str, vector: np.ndarray, spec: Dict):
        # TODO: append to docs/vectors/names and register spec in self.metadata[true_name]
        raise NotImplementedError

    def search(self, query_vec: np.ndarray, k: int) -> List[str]:
        # TODO: cosine-similarity rank self.vectors vs query_vec; return top-k TRUE tool names
        raise NotImplementedError

### S2 . Pre-retrieval enrichment + indexing

**What:** each tool becomes a 5-component document - spaced name, long description, argument
schema, LLM-generated *synthetic questions* it answers, and *key topics/intents* - then embedded.
**Maps to:** enriching our terse catalog entries so an advisor's natural query lands near the right tool.

In [ ]:
def space_name(name: str) -> str:
    # TODO: 'rebalance_portfolio' -> 'rebalance portfolio' (split underscores/camelCase)
    raise NotImplementedError

def generate_synthetic_questions(tool: Dict, llm=llmOpenAI, n: int = 3) -> List[str]:
    # TODO: prompt the LLM for n diverse user questions this tool would answer (reverse-HyDE)
    raise NotImplementedError

def generate_key_topics(tool: Dict, llm=llmOpenAI) -> List[str]:
    # TODO: prompt the LLM for key topics/intents, informed by name+description+synthetic Qs
    raise NotImplementedError

def enrich(tool: Dict) -> str:
    # TODO: concatenate the 5 components into one enriched document string
    raise NotImplementedError

def build_kb(catalog: List[Dict], embedder=embedder) -> 'ToolshedKB':
    kb = ToolshedKB(embedder)
    # TODO: for each tool -> enrich() -> embed -> kb.add(true_name, doc, vector, spec)
    raise NotImplementedError

# kb = build_kb(TOOL_CATALOG)

### S3 . Intra-retrieval - query rewrite

**What:** clean the raw user query (typos, pronouns, conciseness), optionally using chat history.
**Maps to:** turning 'rebalance it and harvest losses' into an explicit, self-contained query.

In [ ]:
def rewrite_query(query: str, chat_history: Optional[List[BaseMessage]] = None, llm=llmOpenAI) -> str:
    # TODO: prompt the LLM to rewrite the query - resolve pronouns, fix typos, keep it concise
    raise NotImplementedError

### S4 . Intra-retrieval - query decomposition / planning

**What:** break a multi-part query into independent sub-steps, each needing its own tool(s).
**Maps to:** 'rebalance AND harvest losses' -> ['rebalance the account', 'harvest tax losses'].

In [ ]:
def decompose_query(query: str, llm=llmOpenAI) -> List[str]:
    # TODO: prompt the LLM to return a list of independent sub-steps (the plan)
    raise NotImplementedError

### S5 . Intra-retrieval - multi-query expansion

**What:** for each sub-step, generate several diverse phrasings (the module is tool-agnostic -
it casts a wide net). **Maps to:** 'harvest tax losses' -> {tax-loss harvesting, realize losses,
offset capital gains, ...} so whatever the right tool is filed under, some phrasing matches it.

In [ ]:
def expand_query(sub_step: str, n: int = 3, llm=llmOpenAI) -> List[str]:
    # TODO: prompt the LLM for n varied phrasings/intents of the sub_step
    raise NotImplementedError

### S6 . Retrieval - query the KB per variant

**What:** embed every query variant and pull its own top-k tool candidates from the KB.
**Maps to:** running each expanded phrasing against ToolshedKB.search and collecting candidates.

In [ ]:
def retrieve(variants: List[str], kb: 'ToolshedKB', k: int = 5, embedder=embedder) -> List[str]:
    # TODO: embed each variant, kb.search(vec, k), union the returned true tool names
    raise NotImplementedError

### S7 . Post-retrieval - rerank + dedupe

**What:** the variants over-fetch and duplicate; an LLM (or cross-encoder) reranker condenses the
candidate pile to the final `top-k`, dropping irrelevant tools and removing duplicates.
**Maps to:** producing the tight toolset actually handed to the agent.

In [ ]:
def rerank_and_dedupe(query: str, candidates: List[str], kb: 'ToolshedKB', top_k: int = 4, llm=llmOpenAI) -> List[str]:
    # TODO: dedupe candidates; prompt the LLM to score/keep the top_k most relevant to `query`
    raise NotImplementedError

### S8 . Post-retrieval - self-reflection (Self-RAG)

**What:** the agent judges whether the final toolset is sufficient; if a needed tool seems missing,
it re-queries the KB (corrective / Self-RAG). **Maps to:** an optional safety net before answering.

In [ ]:
def reflect_sufficiency(query: str, final_tools: List[str], kb: 'ToolshedKB', llm=llmOpenAI) -> List[str]:
    # TODO: ask the LLM 'are these tools enough for the query?'; if not, retrieve more and merge
    raise NotImplementedError

### S9 . tool-M x top-k trade-off + Recall@k

**What:** the paper's cost dial. Measure Recall@k (did the golden tools land in the retrieved set?)
as you vary catalog size (`tool-M`) and threshold (`top-k`). **Maps to:** scoring our GOLDEN pairings.

In [ ]:
def recall_at_k(kb: 'ToolshedKB', golden: List[Dict], k: int) -> float:
    # TODO: for each golden item, run intra+retrieve+rerank to k tools;
    #       recall = |retrieved & golden| / |golden|; return the mean across items
    raise NotImplementedError

# TODO: sweep k in [1, 2, 4, 8] and print Recall@k to see the accuracy/cost trade-off
# for k in (1, 2, 4, 8):
#     print(k, recall_at_k(kb, GOLDEN, k))

### S10 . Final agent tool-calling

**What:** equip the agent with ONLY the final top-k tools (the paper stops here; we go end-to-end).
**Maps to:** binding the selected callables and letting the agent answer the advisor query.

In [ ]:
def make_callables(tool_names: List[str], kb: 'ToolshedKB') -> List:
    # TODO: resolve each true name via kb.metadata into a @tool-decorated callable
    raise NotImplementedError

def answer(query: str, final_tools: List[str], kb: 'ToolshedKB', llm=llmOpenAI) -> AIMessage:
    # TODO: llm.bind_tools(make_callables(final_tools, kb)).invoke([HumanMessage(query)])
    raise NotImplementedError

## S11 . Wire-up - assemble the phases into one LangGraph

Connect pre-retrieval (done at build time) -> intra-retrieval -> retrieval -> post-retrieval ->
self-reflection -> answer into a single runnable graph.

In [ ]:
class ToolshedState(TypedDict):
    query: str
    rewritten: str
    sub_steps: List[str]
    variants: List[str]
    candidates: List[str]
    final_tools: List[str]
    messages: Annotated[List[BaseMessage], add_messages]

# TODO: define node functions wrapping S3-S10, e.g.
# def n_rewrite(state):    return {'rewritten': rewrite_query(state['query'])}
# def n_decompose(state):  return {'sub_steps': decompose_query(state['rewritten'])}
# def n_expand(state):     ...
# def n_retrieve(state):   ...
# def n_rerank(state):     ...
# def n_reflect(state):    ...
# def n_answer(state):     ...

# TODO: build the graph
# g = StateGraph(ToolshedState)
# g.add_node('rewrite', n_rewrite); g.add_node('decompose', n_decompose); ...
# g.add_edge(START, 'rewrite'); ...; g.add_edge('answer', END)
# app = g.compile()
# app.invoke({'query': GOLDEN[0]['query'], 'messages': []})

## Validation - run end-to-end, then tick every box

Run the compiled graph on each `GOLDEN` query and confirm each aspect works. Completing this
checklist == you've reconstructed the whole paper.

- [ ] **S1** KB stores enriched docs and resolves names -> callables
- [ ] **S2** Each tool enriched with all 5 components and embedded
- [ ] **S3** Query rewrite resolves pronouns / cleans the query
- [ ] **S4** Multi-part query decomposes into independent sub-steps
- [ ] **S5** Each sub-step expands into diverse phrasings
- [ ] **S6** Retrieval returns top-k candidates per variant
- [ ] **S7** Rerank + dedupe produce a tight final top-k
- [ ] **S8** Self-reflection fetches a missing tool when one is absent
- [ ] **S9** Recall@k computed; you can see it rise with k (the trade-off)
- [ ] **S10** Agent is bound ONLY the final top-k tools and answers
- [ ] **S11** All phases run as one LangGraph on every GOLDEN query